In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../../data/road_fine.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "dismissal": "string",
        "vehicleClass": "string",
        "notificationType": "string",
        "lastSent": "string",
        "amount": "float32",
        "totalPaymentAmount": "float32",
        "article": "float32",
        "points": "float32",
        "expense": "float32",
        "paymentAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,A1,2006-07-24,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
1,A1,2006-12-05,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11577600.0,0.0,NA
2,A100,2006-08-02,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
3,A100,2006-12-12,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11404800.0,0.0,NA
4,A100,2007-01-15,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,2937600.0,0.0,NA
5,A100,2007-03-16,71.5,0.0,Add penalty,NA,0.0,NA,complete,NA,NA,0.0,0.0,5184000.0,0.0,NA
6,A100,2009-03-30,0.0,0.0,Send for Credit Collection,NA,0.0,NA,complete,NA,NA,0.0,0.0,64368000.0,0.0,NA
7,A10000,2007-03-09,36.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
8,A10000,2007-07-17,0.0,0.0,Send Fine,NA,13.0,NA,complete,NA,NA,0.0,0.0,11232000.0,0.0,NA
9,A10000,2007-08-02,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,1382400.0,0.0,NA


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['amount', 'article', 'concept:name', 'dismissal', 'expense', 'lastSent', 'lifecycle:transition', 'notificationType', 'org:resource', 'paymentAmount', 'points', 'time_delta', 'totalPaymentAmount', 'vehicleClass']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 8726400.00]                       2160000.0000 quantile_derived    
amount                         continuous     event    yes    [24.00, 80.00]                           6.7000     quantile_derived    
totalPaymentAmount             continuous     event

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    path = "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'}]

In [16]:
engine.branching_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine',
  'Send for Credit Collection'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'},
 {'Appeal to Judge', 'Insert Fine Notification', 'Send Fine'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/road_fine-cf_seed777_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/310 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,A19080,5,1,4,0.273542,0.447085,0.100,0.500000,0.384615,...,1.673573,0.384615,0.273542,0.100,0.447085,0.500000,0.515415,0.515415,0.999977,0.999977
1,0,S106832,6,1,5,0.258190,0.416381,0.100,0.541667,0.333333,...,1.850282,0.333333,0.258190,0.100,0.416381,0.541667,0.717092,0.717092,0.999999,0.999999
2,0,S160509,7,1,4,0.228224,0.456448,0.000,0.416667,0.176471,...,1.727411,0.176471,0.228224,0.000,0.456448,0.416667,0.906050,0.906050,1.000000,1.000000
3,1,S71772,3,1,2,0.126733,0.253467,0.000,0.333333,0.000000,...,0.460067,0.000000,0.126733,0.000,0.253467,0.333333,0.000000,0.000000,0.000000,0.000000
4,1,V6556,3,1,2,0.146292,0.292583,0.000,0.291667,0.000000,...,0.437958,0.000000,0.146292,0.000,0.292583,0.291667,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,29,S54626,9,1,4,0.220747,0.366494,0.075,0.458333,0.619048,...,1.298128,0.619048,0.220747,0.075,0.366494,0.458333,0.000000,0.000000,0.000000,0.000000
222,30,N53905,9,1,4,0.163109,0.326217,0.000,0.354167,0.619048,...,1.136323,0.619048,0.163109,0.000,0.326217,0.354167,0.000000,0.000000,0.000000,0.000000
223,30,A373,9,1,4,0.172785,0.345569,0.000,0.364583,0.619048,...,1.156416,0.619048,0.172785,0.000,0.345569,0.364583,0.000000,0.000000,0.000000,0.000000
224,30,N56915,9,1,4,0.249295,0.398590,0.100,0.479167,0.619048,...,1.347510,0.619048,0.249295,0.100,0.398590,0.479167,0.000000,0.000000,0.000000,0.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/310 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,A19080,5,1,4,0.347113,0.494226,0.200,0.541667,0.000000,...,0.888780,0.000000,0.347113,0.200,0.494226,0.541667,0.00000,0.0,0.0,0.0
1,0,S106832,6,1,5,0.373228,0.446456,0.300,0.541667,0.133333,...,1.092868,0.133333,0.373228,0.300,0.446456,0.541667,0.04464,0.0,0.0,0.0
2,0,S160509,7,1,4,0.509384,0.618769,0.400,0.625000,0.352941,...,1.487326,0.352941,0.509384,0.400,0.618769,0.625000,0.00000,0.0,0.0,0.0
3,1,S71772,3,1,2,0.196752,0.293503,0.100,0.416667,0.000000,...,0.613418,0.000000,0.196752,0.100,0.293503,0.416667,0.00000,0.0,0.0,0.0
4,1,V6556,3,1,2,0.146292,0.292585,0.000,0.375000,0.000000,...,0.521292,0.000000,0.146292,0.000,0.292585,0.375000,0.00000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,29,S54626,9,1,4,0.246834,0.443668,0.050,0.479167,0.619048,...,1.345048,0.619048,0.246834,0.050,0.443668,0.479167,0.00000,0.0,0.0,0.0
222,30,N53905,9,1,4,0.203637,0.407275,0.000,0.468750,0.619048,...,1.291435,0.619048,0.203637,0.000,0.407275,0.468750,0.00000,0.0,0.0,0.0
223,30,A373,9,1,4,0.214282,0.428564,0.000,0.468750,0.619048,...,1.302079,0.619048,0.214282,0.000,0.428564,0.468750,0.00000,0.0,0.0,0.0
224,30,N56915,9,1,4,0.330829,0.486658,0.175,0.541667,0.619048,...,1.491543,0.619048,0.330829,0.175,0.486658,0.541667,0.00000,0.0,0.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()